In [2]:
!pip install prophet

   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.1 MB 8.4 MB/s eta 0:00:02
   ------------- -------------------------- 4.2/12.1 MB 10.3 MB/s eta 0:00:01
   ---------------------- ----------------- 6.8/12.1 MB 11.7 MB/s eta 0:00:01
   ------------------------------ --------- 9.2/12.1 MB 11.1 MB/s eta 0:00:01
   -------------------------------------- - 11.8/12.1 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 12.1/12.1 MB 10.9 MB/s  0:00:01
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 10.6 MB/s  0:00:00

   ---------------------------------------- 0/5 [stanio]
   ---------------------------------------- 0/5 [stanio]
   -------- ------------------------------- 1/5 [importlib_resources]
   -------- ------------------------------- 1/5 [importlib_resources]
   -------- ------------------------------- 1/5 [importlib_resourc

In [4]:
from prophet import Prophet
print("Prophet installed successfully")

Prophet installed successfully


In [5]:
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt

In [6]:
df = pd.read_csv("Reatil_data_final.csv")
df.head()

,Date,Customer_ID,Product,Category,Quantity,Price,Sales,Gender,Age,Year,Month_Name,Region,Age_Group,Customer_Type,Payment_Method,Sales_Channel
0,01-01-2023,1051,Mobile,Electronics,3,38828,116484,Male,24,2023,January,East,18–25,Returning,Debit Card,Online
1,01-01-2023,1092,Bag,Accessories,3,16392,49176,Male,60,2023,January,West,46–60,Returning,Debit Card,Online
2,02-01-2023,1014,Laptop,Electronics,3,47799,143397,Male,41,2023,January,West,36–45,New,UPI,Offline
3,03-01-2023,1071,Shoes,Fashion,1,17573,17573,Female,21,2023,January,Central,18–25,New,Credit Card,Offline
4,03-01-2023,1060,Headphones,Electronics,4,37268,149072,Male,45,2023,January,South,36–45,New,Cash,Offline


In [7]:
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

In [8]:
prophet_df = df[['Date','Sales']]
prophet_df = prophet_df.rename(columns={
    'Date':'ds',
    'Sales':'y'
})
prophet_df.head()

,ds,y
0,2023-01-01,116484
1,2023-01-01,49176
2,2023-01-02,143397
3,2023-01-03,17573
4,2023-01-03,149072


In [9]:
prophet_df = prophet_df.groupby('ds').sum().reset_index()

In [10]:
model = Prophet()

model.fit(prophet_df)

00:31:39 - cmdstanpy - INFO - Chain [1] start processing
00:31:40 - cmdstanpy - INFO - Chain [1] done processing


In [11]:
future = model.make_future_dataframe(periods=365)

In [12]:
forecast = model.predict(future)

In [15]:
forecast_data = forecast[['ds','yhat']]

In [16]:
forecast_data.tail()

,ds,yhat
1091,2025-12-27,100754.369605
1092,2025-12-28,114459.903967
1093,2025-12-29,113503.538576
1094,2025-12-30,122574.094153
1095,2025-12-31,110498.171002


In [17]:
predicted_sales = forecast_data[['ds','yhat']].copy()

predicted_sales = predicted_sales.rename(columns={
    'ds':'Date',
    'yhat':'Sales'
})

In [18]:
predicted_sales['Date'] = pd.to_datetime(predicted_sales['Date'])

In [19]:
last_date = df['Date'].max()

predicted_sales = predicted_sales[predicted_sales['Date'] > last_date]

In [20]:
final_data = pd.concat([df[['Date','Sales']], predicted_sales], ignore_index=True)

In [21]:
final_data.to_csv("retail_sales_with_predictions.csv", index=False)